# Hands-On Projects with LangGraph\n\n## Tutorials Covered:\n1. Customer Support Chatbot\n2. Research Assistant\n3. Document Analysis Pipeline\n4. Task Management System\n5. Content Generation Workflow

In [1]:
# Install required packages for hands-on projects
%pip install -q langgraph langchain-openai python-dotenv tavily-python

In [2]:
# Import necessary libraries for hands-on projects
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, List, Dict, Any
import operator
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langchain.schema import AIMessage, HumanMessage, SystemMessage
from langchain.tools import tool
from langchain.agents import tool
import json
import requests
import asyncio
import logging
from datetime import datetime

# Load environment variables
load_dotenv()

# Initialize LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

## 1. Customer Support Chatbot\n\nLearning objectives:\n- Build a contextual customer support bot\n- Implement intent recognition and routing\n- Handle escalation scenarios

In [3]:
# Tutorial 1: Customer Support Chatbot

# Define state for customer support chatbot
class SupportBotState(TypedDict):
    user_input: str
    chat_history: List[Dict[str, str]]
    current_intent: str
    response: str
    needs_escalation: bool
    user_context: Dict[str, Any]
    ticket_id: str
    resolution_status: str

# Customer database simulation
customer_database = {
    'user123': {
        'name': 'John Doe',
        'email': 'john@example.com',
        'premium': True,
        'past_tickets': ['TKT-001', 'TKT-002'],
        'account_status': 'active'
    },
    'user456': {
        'name': 'Jane Smith',
        'email': 'jane@example.com',
        'premium': False,
        'past_tickets': ['TKT-003'],
        'account_status': 'active'
    }
}

# Intent classification node
def classify_intent(state: SupportBotState):
    user_input = state['user_input'].lower()
    
    if any(word in user_input for word in ['billing', 'payment', 'charge', 'invoice', 'refund']):
        intent = 'billing_issue'
    elif any(word in user_input for word in ['login', 'password', 'access', 'account', 'signin', 'signup']):
        intent = 'account_issue'
    elif any(word in user_input for word in ['bug', 'error', 'crash', 'not working', 'broken']):
        intent = 'technical_issue'
    elif any(word in user_input for word in ['cancel', 'delete', 'terminate', 'unsubscribe']):
        intent = 'cancellation_request'
    elif any(word in user_input for word in ['thank', 'thanks', 'great', 'helpful']):
        intent = 'positive_feedback'
    else:
        intent = 'general_inquiry'
    
    return {'current_intent': intent}

# Retrieve user context
def retrieve_user_context(state: SupportBotState):
    # In a real implementation, this would fetch from a database
    # For simulation, we'll use a hardcoded user ID
    user_id = state.get('user_context', {}).get('user_id', 'user123')
    user_data = customer_database.get(user_id, {
        'name': 'Valued Customer',
        'email': 'customer@example.com',
        'premium': False,
        'past_tickets': [],
        'account_status': 'active'
    })
    
    return {'user_context': user_data}

# Handle billing issues
def handle_billing_issue(state: SupportBotState):
    user_context = state['user_context']
    
    response = f"Hello {user_context['name']}, I understand you're having billing issues. "
    response += "Our billing department can help you with that. "
    
    # Premium users get priority
    if user_context['premium']:
        response += "As a premium member, we'll prioritize your request. "
    
    response += "Could you please provide more details about the billing issue you're experiencing?"
    
    # Update chat history
    new_history = state.get('chat_history', []) + [
        {'role': 'user', 'content': state['user_input']},
        {'role': 'assistant', 'content': response}
    ]
    
    return {
        'response': response,
        'chat_history': new_history,
        'resolution_status': 'in_progress'
    }

# Handle account issues
def handle_account_issue(state: SupportBotState):
    user_context = state['user_context']
    
    response = f"Hello {user_context['name']}, I can help you with your account issue. "
    response += "Let me look up your account information. "
    
    # Check account status
    if user_context['account_status'] != 'active':
        response += "I notice your account isn't active. This might be why you're having access issues. "
    
    response += "Could you please specify what account-related issue you're facing?"
    
    # Update chat history
    new_history = state.get('chat_history', []) + [
        {'role': 'user', 'content': state['user_input']},
        {'role': 'assistant', 'content': response}
    ]
    
    return {
        'response': response,
        'chat_history': new_history,
        'resolution_status': 'in_progress'
    }

# Handle technical issues
def handle_technical_issue(state: SupportBotState):
    user_context = state['user_context']
    
    response = f"Hello {user_context['name']}, I'm sorry to hear you're experiencing technical difficulties. "
    response += "I'll try to help you troubleshoot this issue. "
    
    # Check if issue is severe enough for escalation
    user_input_lower = state['user_input'].lower()
    if any(severity_word in user_input_lower for severity_word in ['crash', 'completely broken', 'can\'t access', 'urgent']):
        response += "This sounds like a critical issue that may require immediate attention. "
        return {
            'response': response,
            'needs_escalation': True,
            'resolution_status': 'escalated'
        }
    
    response += "Can you provide more details about the error message or the specific problem you're encountering?"
    
    # Update chat history
    new_history = state.get('chat_history', []) + [
        {'role': 'user', 'content': state['user_input']},
        {'role': 'assistant', 'content': response}
    ]
    
    return {
        'response': response,
        'chat_history': new_history,
        'resolution_status': 'in_progress'
    }

# Handle cancellation requests
def handle_cancellation_request(state: SupportBotState):
    user_context = state['user_context']
    
    response = f"Hello {user_context['name']}, I'm sorry to hear you're considering canceling your subscription. "
    response += "Before proceeding, I'd like to understand why you want to cancel. "
    response += "Perhaps we can resolve the issue that's causing your dissatisfaction. "
    
    response += "Could you share what's prompting this decision?"
    
    # Update chat history
    new_history = state.get('chat_history', []) + [
        {'role': 'user', 'content': state['user_input']},
        {'role': 'assistant', 'content': response}
    ]
    
    return {
        'response': response,
        'chat_history': new_history,
        'resolution_status': 'in_progress'
    }

# Handle general inquiries
def handle_general_inquiry(state: SupportBotState):
    user_context = state['user_context']
    
    response = f"Hello {user_context['name']}, thanks for reaching out. "
    response += f"I received your inquiry: '{state['user_input'][:50]}...'. "
    response += "Let me address your question. "
    
    # For demo purposes, we'll give a generic response
    response += "I've noted your inquiry and will provide a detailed response shortly. "
    response += "Is there anything else I can assist you with today?"
    
    # Update chat history
    new_history = state.get('chat_history', []) + [
        {'role': 'user', 'content': state['user_input']},
        {'role': 'assistant', 'content': response}
    ]
    
    return {
        'response': response,
        'chat_history': new_history,
        'resolution_status': 'in_progress'
    }

# Route based on intent
def route_by_intent(state: SupportBotState):
    intent = state.get('current_intent', 'general_inquiry')
    
    if intent == 'billing_issue':
        return 'handle_billing_issue'
    elif intent == 'account_issue':
        return 'handle_account_issue'
    elif intent == 'technical_issue':
        return 'handle_technical_issue'
    elif intent == 'cancellation_request':
        return 'handle_cancellation_request'
    else:
        return 'handle_general_inquiry'

# Escalate to human agent
def escalate_to_human(state: SupportBotState):
    # Generate a ticket ID
    import random
    ticket_id = f"TKT-{random.randint(1000, 9999)}"
    
    response = "I've escalated your issue to our technical team. "
    response += f"Your ticket ID is {ticket_id}. A human agent will contact you within 24 hours. "
    response += "Thank you for your patience."
    
    # Update chat history
    new_history = state.get('chat_history', []) + [
        {'role': 'assistant', 'content': response}
    ]
    
    return {
        'response': response,
        'ticket_id': ticket_id,
        'chat_history': new_history,
        'resolution_status': 'escalated_to_human'
    }

# Build the customer support chatbot graph
def create_support_bot_graph():
    workflow = StateGraph(SupportBotState)
    
    # Add nodes
    workflow.add_node('classify_intent', classify_intent)
    workflow.add_node('retrieve_user_context', retrieve_user_context)
    workflow.add_node('handle_billing_issue', handle_billing_issue)
    workflow.add_node('handle_account_issue', handle_account_issue)
    workflow.add_node('handle_technical_issue', handle_technical_issue)
    workflow.add_node('handle_cancellation_request', handle_cancellation_request)
    workflow.add_node('handle_general_inquiry', handle_general_inquiry)
    workflow.add_node('escalate_to_human', escalate_to_human)
    
    # Set entry point
    workflow.set_entry_point('retrieve_user_context')
    
    # Add edges
    workflow.add_edge('retrieve_user_context', 'classify_intent')
    workflow.add_conditional_edges(
        'classify_intent',
        route_by_intent,
        {
            'handle_billing_issue': 'handle_billing_issue',
            'handle_account_issue': 'handle_account_issue',
            'handle_technical_issue': 'handle_technical_issue',
            'handle_cancellation_request': 'handle_cancellation_request',
            'handle_general_inquiry': 'handle_general_inquiry'
        }
    )
    
    # Add conditional edges for escalation
    def should_escalate(state: SupportBotState):
        if state.get('needs_escalation', False):
            return 'escalate_to_human'
        else:
            return END
    
    workflow.add_conditional_edges(
        'handle_technical_issue',
        should_escalate,
        {
            'escalate_to_human': 'escalate_to_human',
            END: END
        }
    )
    
    # Add edges from other handlers to end
    workflow.add_edge('handle_billing_issue', END)
    workflow.add_edge('handle_account_issue', END)
    workflow.add_edge('handle_cancellation_request', END)
    workflow.add_edge('handle_general_inquiry', END)
    workflow.add_edge('escalate_to_human', END)
    
    return workflow.compile()

In [4]:
# Test the customer support chatbot
support_bot_graph = create_support_bot_graph()

# Example: User reporting an account issue
initial_state_support = {
    'user_input': "I'm having trouble logging into my account and keep getting an error message",
    'chat_history': [],
    'current_intent': '',
    'response': '',
    'needs_escalation': False,
    'user_context': {'user_id': 'user123'},
    'ticket_id': '',
    'resolution_status': ''
}

result_support = support_bot_graph.invoke(initial_state_support)
print("Final support bot state:", result_support)

Final support bot state: {'user_input': 'I\'m having trouble logging into my account and keep getting an error message', 'chat_history': [{'role': 'user', 'content': "I'm having trouble logging into my account and keep getting an error message"}, {'role': 'assistant', 'content': "Hello John Doe, I can help you with your account issue. Let me look up your account information. Could you please specify what account-related issue you're facing?"}], 'current_intent': 'account_issue', 'response': "Hello John Doe, I can help you with your account issue. Let me look up your account information. Could you please specify what account-related issue you're facing?", 'needs_escalation': False, 'user_context': {'name': 'John Doe', 'email': 'john@example.com', 'premium': True, 'past_tickets': ['TKT-001', 'TKT-002'], 'account_status': 'active'}, 'ticket_id': '', 'resolution_status': 'in_progress'}


## 2. Research Assistant\n\nLearning objectives:\n- Create an AI researcher that can gather information\n- Implement web search and content synthesis\n- Build knowledge aggregation capabilities

In [5]:
# Tutorial 2: Research Assistant

# For this example, we'll simulate a research assistant
# In a real implementation, you would integrate with search APIs

# Define state for research assistant
class ResearchAssistantState(TypedDict):
    research_query: str
    search_queries: List[str]
    search_results: List[Dict[str, Any]]
    synthesized_findings: str
    sources: List[str]
    research_steps: List[str]
    final_report: str
    credibility_score: float
    fact_checks: List[Dict[str, Any]]

# Function to simulate web search
def simulate_web_search(query: str) -> List[Dict[str, str]]:
    # In a real implementation, this would call a search API
    # For simulation, we'll return mock results
    mock_results = [
        {"title": f"Article 1 about {query}", "url": f"https://example.com/{query}/article1", "content": f"This article discusses the main aspects of {query}. It covers important points and provides valuable insights."},
        {"title": f"Study on {query}", "url": f"https://example.com/{query}/study", "content": f"Recent study shows significant findings related to {query}. The research methodology was robust and results are considered reliable."},
        {"title": f"Overview of {query}", "url": f"https://example.com/{query}/overview", "content": f"Comprehensive overview of {query} including historical context, current developments, and future projections."}
    ]
    return mock_results

# Plan the research approach
def plan_research_approach(state: ResearchAssistantState):
    query = state['research_query']
    
    # Generate search queries based on the main query
    queries = [
        f"what is {query}",
        f"{query} recent developments",
        f"{query} challenges and opportunities",
        f"{query} future trends"
    ]
    
    research_plan = f"Research plan for '{query}':\n1. Define scope and objectives\n2. Search for foundational information\n3. Look for recent developments\n4. Identify challenges and opportunities\n5. Synthesize findings"
    
    return {
        'search_queries': queries,
        'research_steps': [research_plan]
    }

# Execute search queries
def execute_searches(state: ResearchAssistantState):
    queries = state['search_queries']
    all_results = []
    all_sources = []
    
    for query in queries:
        # Simulate web search
        results = simulate_web_search(query)
        all_results.extend(results)
        
        # Collect sources
        sources = [r['url'] for r in results]
        all_sources.extend(sources)
    
    search_step = f"Executed {len(queries)} search queries and collected {len(all_results)} results"
    
    return {
        'search_results': all_results,
        'sources': all_sources,
        'research_steps': state.get('research_steps', []) + [search_step]
    }

# Evaluate source credibility
def evaluate_credibility(state: ResearchAssistantState):
    results = state['search_results']
    
    # In a real implementation, this would analyze source domain authority, publication date, etc.
    # For simulation, we'll assign a random score
    import random
    credibility_score = round(random.uniform(0.6, 0.95), 2)
    
    # Perform basic fact checks
    fact_checks = []
    for i, result in enumerate(results[:3]):  # Check first 3 results
        fact_checks.append({
            'statement': result['content'][:100] + '...',
            'source': result['url'],
            'credibility': 'high' if credibility_score > 0.8 else 'medium'
        })
    
    evaluation_step = f"Evaluated credibility of sources with overall score: {credibility_score}"
    
    return {
        'credibility_score': credibility_score,
        'fact_checks': fact_checks,
        'research_steps': state.get('research_steps', []) + [evaluation_step]
    }

# Synthesize the gathered information
def synthesize_findings(state: ResearchAssistantState):
    results = state['search_results']
    query = state['research_query']
    
    # Extract key points from results
    key_points = []
    for result in results[:5]:  # Take first 5 results
        key_points.append(result['content'][:150] + '...')
    
    synthesized_content = f"Based on research about '{query}', key findings include: {'; '.join(key_points[:3])}"
    
    synthesis_step = f"Synthesized information from {len(results)} sources"
    
    return {
        'synthesized_findings': synthesized_content,
        'research_steps': state.get('research_steps', []) + [synthesis_step]
    }

# Generate final research report
def generate_final_report(state: ResearchAssistantState):
    query = state['research_query']
    synthesized = state['synthesized_findings']
    sources = state['sources'][:5]  # Limit to first 5 sources
    credibility = state['credibility_score']
    
    report = f"Research Report on '{query}':\n{synthesized}\n\nSources consulted: {sources}\n\nCredibility Score: {credibility}"
    
    report_step = f"Generated final report on {query} with credibility score {credibility}"
    
    return {
        'final_report': report,
        'research_steps': state.get('research_steps', []) + [report_step]
    }

# Build the research assistant graph
def create_research_assistant_graph():
    workflow = StateGraph(ResearchAssistantState)
    
    # Add nodes
    workflow.add_node('plan_research_approach', plan_research_approach)
    workflow.add_node('execute_searches', execute_searches)
    workflow.add_node('evaluate_credibility', evaluate_credibility)
    workflow.add_node('synthesize_findings', synthesize_findings)
    workflow.add_node('generate_final_report', generate_final_report)
    
    # Set entry point
    workflow.set_entry_point('plan_research_approach')
    
    # Add edges
    workflow.add_edge('plan_research_approach', 'execute_searches')
    workflow.add_edge('execute_searches', 'evaluate_credibility')
    workflow.add_edge('evaluate_credibility', 'synthesize_findings')
    workflow.add_edge('synthesize_findings', 'generate_final_report')
    workflow.add_edge('generate_final_report', END)
    
    return workflow.compile()

In [6]:
# Test the research assistant
research_assistant_graph = create_research_assistant_graph()

# Example: Research on renewable energy adoption trends
initial_state_research = {
    'research_query': 'renewable energy adoption trends',
    'search_queries': [],
    'search_results': [],
    'synthesized_findings': '',
    'sources': [],
    'research_steps': [],
    'final_report': '',
    'credibility_score': 0.0,
    'fact_checks': []
}

result_research = research_assistant_graph.invoke(initial_state_research)
print("Final research assistant state:", result_research)

Final research assistant state: {'research_query': 'renewable energy adoption trends', 'search_queries': ['what is renewable energy adoption trends', 'renewable energy adoption trends recent developments', 'renewable energy adoption trends challenges and opportunities', 'renewable energy adoption trends future trends'], 'search_results': [{'title': 'Article 1 about what is renewable energy adoption trends', 'url': 'https://example.com/what is renewable energy adoption trends/article1', 'content': 'This article discusses the main aspects of what is renewable energy adoption trends. It covers important points and provides valuable insights.'}, {'title': 'Study on what is renewable energy adoption trends', 'url': 'https://example.com/what is renewable energy adoption trends/study', 'content': 'Recent study shows significant findings related to what is renewable energy adoption trends. The research methodology was robust and results are considered reliable.'}, {'title': 'Overview of what i

## 3. Document Analysis Pipeline\n\nLearning objectives:\n- Build a system for processing and analyzing documents\n- Extract structured information from unstructured text\n- Classify and route documents automatically

In [7]:
# Tutorial 3: Document Analysis Pipeline

# Define state for document analysis pipeline
class DocumentAnalysisState(TypedDict):
    documents: List[Dict[str, Any]]
    extracted_content: List[Dict[str, Any]]
    document_types: List[str]
    classification_results: List[Dict[str, Any]]
    extracted_entities: List[Dict[str, Any]]
    processed_documents: List[Dict[str, Any]]
    routing_decisions: List[Dict[str, str]]
    analysis_summary: str
    quality_scores: List[float]

# Simulate document ingestion
def ingest_documents(state: DocumentAnalysisState):
    # In a real implementation, this would read from files or an API
    sample_docs = [
        {
            'id': 'doc1',
            'content': 'INVOICE #INV-2023-001. Date: 2023-05-15. Client: Acme Corp. Amount: $1,250.00. Services: Software consulting for Q2 project.',
            'source': 'email_attachment',
            'metadata': {'sender': 'billing@acmecorp.com', 'received_date': '2023-05-15'}
        },
        {
            'id': 'doc2',
            'content': 'CONFIDENTIAL CONTRACT. Agreement between TechSolutions Inc. and Global Enterprises. Effective date: 2023-06-01. Duration: 2 years. Value: $500,000.',
            'source': 'scanned_pdf',
            'metadata': {'department': 'legal', 'priority': 'high'}
        },
        {
            'id': 'doc3',
            'content': 'EMPLOYEE PERFORMANCE REVIEW. Employee: John Smith. Period: Jan 2023 - Mar 2023. Rating: Exceeds Expectations. Strengths: Leadership, Innovation.',
            'source': 'hr_system',
            'metadata': {'employee_id': 'emp001', 'reviewer': 'manager1'}
        }
    ]
    
    return {
        'documents': sample_docs
    }

# Classify document types
def classify_documents(state: DocumentAnalysisState):
    documents = state['documents']
    classifications = []
    doc_types = []
    quality_scores = []
    
    for doc in documents:
        content = doc['content'].upper()
        
        if 'INVOICE' in content or 'BILL' in content or 'AMOUNT:' in content:
            doc_type = 'invoice'
            confidence = 0.95
            quality_score = 0.9
        elif 'CONTRACT' in content or 'AGREEMENT' in content or 'TERMS:' in content:
            doc_type = 'contract'
            confidence = 0.90
            quality_score = 0.85
        elif 'REVIEW' in content or 'PERFORMANCE' in content or 'EMPLOYEE:' in content:
            doc_type = 'performance_review'
            confidence = 0.85
            quality_score = 0.8
        elif 'REPORT' in content or 'ANALYSIS' in content:
            doc_type = 'report'
            confidence = 0.80
            quality_score = 0.75
        else:
            doc_type = 'general_document'
            confidence = 0.60
            quality_score = 0.6
        
        classification = {
            'document_id': doc['id'],
            'document_type': doc_type,
            'confidence': confidence,
            'keywords': [word for word in ['INVOICE', 'CONTRACT', 'REVIEW', 'REPORT', 'BILL', 'AGREEMENT'] if word in content]
        }
        
        classifications.append(classification)
        doc_types.append(doc_type)
        quality_scores.append(quality_score)
    
    return {
        'classification_results': classifications,
        'document_types': doc_types,
        'quality_scores': quality_scores
    }

# Extract key entities from documents
def extract_entities(state: DocumentAnalysisState):
    documents = state['documents']
    extracted_entities_list = []
    extracted_content_list = []
    
    for doc in documents:
        content = doc['content']
        
        # Extract dates
        import re
        dates = re.findall(r'\d{4}-\d{2}-\d{2}', content)
        
        # Extract monetary amounts
        amounts = re.findall(r'\$[\d,]+\.\d{2}', content)
        
        # Extract entities based on document type
        if 'invoice' in content.lower():
            invoice_num = re.findall(r'INV-\d{4}-\d+', content)
            client = re.findall(r'Client: ([^.]+)', content)
            entities = {
                'invoice_number': invoice_num[0] if invoice_num else None,
                'client': client[0] if client else None,
                'amount': amounts[0] if amounts else None,
                'dates': dates
            }
        elif 'contract' in content.lower():
            parties = re.findall(r'between ([^.]*) and ([^.]*[.])', content)
            entities = {
                'parties': parties[0] if parties else None,
                'effective_date': dates[0] if dates else None,
                'amount': amounts[0] if amounts else None,
                'duration': re.findall(r'Duration: ([^.]+)', content)[0] if re.findall(r'Duration: ([^.]+)', content) else None
            }
        elif 'review' in content.lower():
            employee = re.findall(r'Employee: ([^.]+)', content)
            entities = {
                'employee': employee[0] if employee else None,
                'rating': re.findall(r'Rating: ([^.]+)', content)[0] if re.findall(r'Rating: ([^.]+)', content) else None,
                'period': re.findall(r'Period: ([^.]+)', content)[0] if re.findall(r'Period: ([^.]+)', content) else None
            }
        else:
            entities = {'free_text': content[:200] + '...'}
        
        extracted_entities_list.append({
            'document_id': doc['id'],
            'entities': entities,
            'raw_content': content
        })
        
        extracted_content_list.append({
            'document_id': doc['id'],
            'summary': content[:100] + '...',
            'extracted_text': content
        })
    
    return {
        'extracted_entities': extracted_entities_list,
        'extracted_content': extracted_content_list
    }

# Validate extracted information
def validate_extraction(state: DocumentAnalysisState):
    entities = state['extracted_entities']
    
    validated_entities = []
    
    for entity_group in entities:
        doc_id = entity_group['document_id']
        entities = entity_group['entities']
        
        # Perform basic validation
        validation_results = {}
        
        for key, value in entities.items():
            if key == 'amount':
                # Validate amount format
                if value:
                    validation_results[key] = re.match(r'\$[\d,]+\.\d{2}', value) is not None
                else:
                    validation_results[key] = False
            elif key == 'dates':
                # Validate date format
                valid_dates = []
                for date in value:
                    try:
                        datetime.strptime(date, '%Y-%m-%d')
                        valid_dates.append(True)
                    except ValueError:
                        valid_dates.append(False)
                validation_results[key] = all(valid_dates) if valid_dates else False
            else:
                validation_results[key] = value is not None and value != ''
        
        validated_entities.append({
            'document_id': doc_id,
            'entities': entities,
            'validations': validation_results,
            'overall_valid': all(validation_results.values())
        })
    
    return {
        'extracted_entities': validated_entities
    }

# Route documents based on type and content
def route_documents(state: DocumentAnalysisState):
    classifications = state['classification_results']
    entities = state['extracted_entities']
    routing_decisions = []
    processed_docs = []
    
    for i, classification in enumerate(classifications):
        doc_type = classification['document_type']
        entity_data = entities[i]['entities']
        
        # Determine routing destination based on document type
        if doc_type == 'invoice':
            destination = 'accounts_payable'
            priority = 'normal'
            if entity_data.get('amount') and float(entity_data['amount'].replace('$', '').replace(',', '')) > 10000:
                priority = 'high'
        elif doc_type == 'contract':
            destination = 'legal_department'
            priority = 'high'
        elif doc_type == 'performance_review':
            destination = 'hr_department'
            priority = 'normal'
        else:
            destination = 'general_inbox'
            priority = 'low'
        
        routing_decision = {
            'document_id': classification['document_id'],
            'destination': destination,
            'priority': priority,
            'reason': f"Document classified as {doc_type}"
        }
        
        routing_decisions.append(routing_decision)
        
        processed_doc = {
            'id': classification['document_id'],
            'type': doc_type,
            'destination': destination,
            'priority': priority,
            'entities': entity_data,
            'status': 'routed',
            'validation_passed': entities[i]['overall_valid']
        }
        
        processed_docs.append(processed_doc)
    
    return {
        'routing_decisions': routing_decisions,
        'processed_documents': processed_docs
    }

# Generate analysis summary
def generate_summary(state: DocumentAnalysisState):
    processed_docs = state['processed_documents']
    quality_scores = state['quality_scores']
    
    # Count document types
    type_counts = {}
    for doc in processed_docs:
        doc_type = doc['type']
        type_counts[doc_type] = type_counts.get(doc_type, 0) + 1
    
    # Calculate average quality score
    avg_quality = sum(quality_scores) / len(quality_scores) if quality_scores else 0
    
    # Count validation passes
    validation_passes = sum(1 for doc in processed_docs if doc['validation_passed'])
    
    summary = f"Document Analysis Summary:\n"
    summary += f"- Total documents processed: {len(processed_docs)}\n"
    summary += f"- Document type distribution: {type_counts}\n"
    summary += f"- Average quality score: {avg_quality:.2f}\n"
    summary += f"- Validation success rate: {validation_passes}/{len(processed_docs)} ({validation_passes/len(processed_docs)*100:.1f}% if processed_docs else 0)%\n"
    
    return {
        'analysis_summary': summary
    }

# Build the document analysis pipeline graph
def create_document_analysis_graph():
    workflow = StateGraph(DocumentAnalysisState)
    
    # Add nodes
    workflow.add_node('ingest_documents', ingest_documents)
    workflow.add_node('classify_documents', classify_documents)
    workflow.add_node('extract_entities', extract_entities)
    workflow.add_node('validate_extraction', validate_extraction)
    workflow.add_node('route_documents', route_documents)
    workflow.add_node('generate_summary', generate_summary)
    
    # Set entry point
    workflow.set_entry_point('ingest_documents')
    
    # Add edges
    workflow.add_edge('ingest_documents', 'classify_documents')
    workflow.add_edge('classify_documents', 'extract_entities')
    workflow.add_edge('extract_entities', 'validate_extraction')
    workflow.add_edge('validate_extraction', 'route_documents')
    workflow.add_edge('route_documents', 'generate_summary')
    workflow.add_edge('generate_summary', END)
    
    return workflow.compile()

In [8]:
# Test the document analysis pipeline
doc_analysis_graph = create_document_analysis_graph()

# Example: Process various documents
initial_state_docs = {
    'documents': [],
    'extracted_content': [],
    'document_types': [],
    'classification_results': [],
    'extracted_entities': [],
    'processed_documents': [],
    'routing_decisions': [],
    'analysis_summary': '',
    'quality_scores': []
}

result_docs = doc_analysis_graph.invoke(initial_state_docs)
print("Final document analysis state:", result_docs)

Final document analysis state: {'documents': [{'id': 'doc1', 'content': 'INVOICE #INV-2023-001. Date: 2023-05-15. Client: Acme Corp. Amount: $1,250.00. Services: Software consulting for Q2 project.', 'source': 'email_attachment', 'metadata': {'sender': 'billing@acmecorp.com', 'received_date': '2023-05-15'}}, {'id': 'doc2', 'content': 'CONFIDENTIAL CONTRACT. Agreement between TechSolutions Inc. and Global Enterprises. Effective date: 2023-06-01. Duration: 2 years. Value: $500,000.', 'source': 'scanned_pdf', 'metadata': {'department': 'legal', 'priority': 'high'}}, {'id': 'doc3', 'content': 'EMPLOYEE PERFORMANCE REVIEW. Employee: John Smith. Period: Jan 2023 - Mar 2023. Rating: Exceeds Expectations. Strengths: Leadership, Innovation.', 'source': 'hr_system', 'metadata': {'employee_id': 'emp001', 'reviewer': 'manager1'}}], 'extracted_content': [{'document_id': 'doc1', 'summary': 'INVOICE #INV-2023-001. Date: 2023-05-15. Client: Acme Corp. Amount: $1,250.00. Services: Software consulting f

## 4. Task Management System\n\nLearning objectives:\n- Create an intelligent task management system\n- Implement task prioritization and scheduling\n- Build progress tracking capabilities

In [9]:
# Tutorial 4: Task Management System

# Define state for task management system
class TaskManagementState(TypedDict):
    incoming_requests: List[Dict[str, Any]]
    tasks: List[Dict[str, Any]]
    assigned_tasks: List[Dict[str, Any]]
    task_priorities: List[Dict[str, Any]]
    task_schedule: List[Dict[str, Any]]
    progress_updates: List[Dict[str, Any]]
    completed_tasks: List[Dict[str, Any]]
    system_notifications: List[str]

# Parse incoming requests
def parse_requests(state: TaskManagementState):
    requests = state['incoming_requests']
    tasks = []
    
    for req in requests:
        # In a real system, this would use NLP to extract task details
        task = {
            'id': f"task_{len(tasks)+1}",
            'title': req.get('description', 'Untitled Task'),
            'description': req.get('description', ''),
            'requester': req.get('requester', 'unknown'),
            'deadline': req.get('deadline', 'not_specified'),
            'complexity': req.get('complexity', 'medium'),  # low, medium, high
            'status': 'created',
            'assigned_to': None,
            'dependencies': req.get('dependencies', [])
        }
        tasks.append(task)
    
    return {
        'tasks': tasks,
        'system_notifications': [f"Parsed {len(requests)} incoming requests into tasks"]
    }

# Prioritize tasks based on various factors
def prioritize_tasks(state: TaskManagementState):
    tasks = state['tasks']
    prioritized_tasks = []
    
    for task in tasks:
        # Calculate priority score based on complexity, deadline, and requester importance
        complexity_factor = {'low': 1, 'medium': 2, 'high': 3}[task['complexity']]
        
        # Deadline factor - urgent tasks get higher priority
        if task['deadline'] == 'not_specified':
            deadline_factor = 1
        else:
            # In a real system, this would calculate urgency based on current date vs deadline
            deadline_factor = 3  # For demo purposes
        
        # Requester importance factor
        requester_importance = 2  # Medium importance for all in demo
        if task['requester'] in ['ceo', 'cto', 'important_client']:
            requester_importance = 3
        
        priority_score = complexity_factor + deadline_factor + requester_importance
        
        prioritized_task = {**task, 'priority_score': priority_score}
        prioritized_tasks.append(prioritized_task)
    
    # Sort by priority score (highest first)
    prioritized_tasks.sort(key=lambda x: x['priority_score'], reverse=True)
    
    return {
        'task_priorities': prioritized_tasks,
        'system_notifications': state.get('system_notifications', []) + [f"Prioritized {len(prioritized_tasks)} tasks"]
    }

# Assign tasks to available resources
def assign_tasks(state: TaskManagementState):
    prioritized_tasks = state['task_priorities']
    
    # Simulate available resources
    resources = [
        {'id': 'res_1', 'name': 'Alice Developer', 'skills': ['coding', 'debugging'], 'availability': 5},
        {'id': 'res_2', 'name': 'Bob Analyst', 'skills': ['analysis', 'reporting'], 'availability': 4},
        {'id': 'res_3', 'name': 'Carol Designer', 'skills': ['design', 'prototyping'], 'availability': 3}
    ]
    
    assigned_tasks = []
    notifications = []
    
    for task in prioritized_tasks:
        # Simple assignment logic - assign to first available resource
        # In a real system, this would consider skills, workload, etc.
        assigned_resource = resources[0]  # Always assign to first resource in demo
        
        assigned_task = {
            **task,
            'assigned_to': assigned_resource['name'],
            'resource_id': assigned_resource['id'],
            'status': 'assigned',
            'estimated_duration': 3  # Days
        }
        assigned_tasks.append(assigned_task)
        
        # Reduce resource availability
        resources[0]['availability'] -= 1
        
        notifications.append(f"Task '{task['title']}' assigned to {assigned_resource['name']}")
    
    return {
        'assigned_tasks': assigned_tasks,
        'system_notifications': state.get('system_notifications', []) + notifications
    }

# Schedule tasks based on priorities and dependencies
def schedule_tasks(state: TaskManagementState):
    assigned_tasks = state['assigned_tasks']
    
    # Create a simple schedule based on priorities
    scheduled_tasks = []
    
    # Sort by priority score (but respect dependencies)
    # For simplicity, we'll just sort by priority in this demo
    sorted_tasks = sorted(assigned_tasks, key=lambda x: x['priority_score'], reverse=True)
    
    current_day = 1
    for i, task in enumerate(sorted_tasks):
        scheduled_task = {
            **task,
            'scheduled_start': current_day,
            'scheduled_end': current_day + task['estimated_duration'],
            'actual_start': None,
            'actual_end': None
        }
        scheduled_tasks.append(scheduled_task)
        
        # Increment day counter
        current_day = scheduled_task['scheduled_end'] + 1
    
    return {
        'task_schedule': scheduled_tasks,
        'system_notifications': state.get('system_notifications', []) + [f"Scheduled {len(scheduled_tasks)} tasks"]
    }

# Simulate progress updates
def update_progress(state: TaskManagementState):
    scheduled_tasks = state['task_schedule']
    progress_updates = []
    completed_tasks = []
    
    for task in scheduled_tasks:
        # Simulate progress - some tasks complete, others in progress
        import random
        progress = random.choice(['not_started', 'in_progress', 'completed'])
        
        progress_update = {
            'task_id': task['id'],
            'task_title': task['title'],
            'progress': progress,
            'update_time': 'now',
            'notes': f'Status update for task {task["title"]}'
        }
        progress_updates.append(progress_update)
        
        # If task is completed, add to completed tasks
        if progress == 'completed':
            completed_task = {**task, 'status': 'completed', 'actual_end': 'today'}
            completed_tasks.append(completed_task)
    
    return {
        'progress_updates': progress_updates,
        'completed_tasks': completed_tasks,
        'system_notifications': state.get('system_notifications', []) + [f"Updated progress for {len(progress_updates)} tasks, {len(completed_tasks)} completed"]
    }

# Generate task summary report
def generate_task_report(state: TaskManagementState):
    scheduled_tasks = state['task_schedule']
    completed_tasks = state['completed_tasks']
    
    total_tasks = len(scheduled_tasks)
    completed_count = len(completed_tasks)
    in_progress_count = len([u for u in state['progress_updates'] if u['progress'] == 'in_progress'])
    pending_count = total_tasks - completed_count - in_progress_count
    
    report = f"Task Management Report:\n"
    report += f"- Total tasks: {total_tasks}\n"
    report += f"- Completed: {completed_count}\n"
    report += f"- In Progress: {in_progress_count}\n"
    report += f"- Pending: {pending_count}\n"
    report += f"- Completion Rate: {completed_count/total_tasks*100:.1f}% if total_tasks > 0 else 0}%\n"
    
    return {
        'system_notifications': state.get('system_notifications', []) + [report]
    }

# Build the task management system graph
def create_task_management_graph():
    workflow = StateGraph(TaskManagementState)
    
    # Add nodes
    workflow.add_node('parse_requests', parse_requests)
    workflow.add_node('prioritize_tasks', prioritize_tasks)
    workflow.add_node('assign_tasks', assign_tasks)
    workflow.add_node('schedule_tasks', schedule_tasks)
    workflow.add_node('update_progress', update_progress)
    workflow.add_node('generate_task_report', generate_task_report)
    
    # Set entry point
    workflow.set_entry_point('parse_requests')
    
    # Add edges
    workflow.add_edge('parse_requests', 'prioritize_tasks')
    workflow.add_edge('prioritize_tasks', 'assign_tasks')
    workflow.add_edge('assign_tasks', 'schedule_tasks')
    workflow.add_edge('schedule_tasks', 'update_progress')
    workflow.add_edge('update_progress', 'generate_task_report')
    workflow.add_edge('generate_task_report', END)
    
    return workflow.compile()

In [10]:
# Test the task management system
task_management_graph = create_task_management_graph()

# Example: Process incoming task requests
initial_state_tasks = {
    'incoming_requests': [
        {
            'description': 'Implement new user authentication system',
            'requester': 'cto',
            'deadline': '2023-06-15',
            'complexity': 'high',
            'dependencies': []
        },
        {
            'description': 'Update documentation for API v2',
            'requester': 'tech_lead',
            'deadline': '2023-06-20',
            'complexity': 'medium',
            'dependencies': ['task_1']
        },
        {
            'description': 'Fix critical bug in payment processing',
            'requester': 'qa_lead',
            'deadline': '2023-06-10',
            'complexity': 'high',
            'dependencies': []
        },
        {
            'description': 'Design dashboard UI components',
            'requester': 'product_manager',
            'deadline': '2023-06-25',
            'complexity': 'medium',
            'dependencies': []
        }
    ],
    'tasks': [],
    'assigned_tasks': [],
    'task_priorities': [],
    'task_schedule': [],
    'progress_updates': [],
    'completed_tasks': [],
    'system_notifications': []
}

result_tasks = task_management_graph.invoke(initial_state_tasks)
print("Final task management state:", result_tasks)

Final task management state: {'incoming_requests': [{'description': 'Implement new user authentication system', 'requester': 'cto', 'deadline': '2023-06-15', 'complexity': 'high', 'dependencies': []}, {'description': 'Update documentation for API v2', 'requester': 'tech_lead', 'deadline': '2023-06-20', 'complexity': 'medium', 'dependencies': ['task_1']}, {'description': 'Fix critical bug in payment processing', 'requester': 'qa_lead', 'deadline': '2023-06-10', 'complexity': 'high', 'dependencies': []}, {'description': 'Design dashboard UI components', 'requester': 'product_manager', 'deadline': '2023-06-25', 'complexity': 'medium', 'dependencies': []}], 'tasks': [{'id': 'task_1', 'title': 'Implement new user authentication system', 'description': 'Implement new user authentication system', 'requester': 'cto', 'deadline': '2023-06-15', 'complexity': 'high', 'status': 'created', 'assigned_to': None, 'dependencies': []}, {'id': 'task_2', 'title': 'Update documentation for API v2', 'descri

## 5. Content Generation Workflow\n\nLearning objectives:\n- Build an automated content generation pipeline\n- Implement quality control mechanisms\n- Create review and approval workflows

In [11]:
# Tutorial 5: Content Generation Workflow

# Define state for content generation workflow
class ContentGenerationState(TypedDict):
    content_request: Dict[str, Any]
    generated_content: str
    content_outline: List[str]
    research_notes: List[str]
    draft_content: str
    quality_score: float
    revision_notes: List[str]
    final_content: str
    approval_status: str
    reviewer_comments: List[str]

# Parse content request
def parse_content_request(state: ContentGenerationState):
    request = state['content_request']
    
    # Extract content requirements
    topic = request.get('topic', 'General Topic')
    length = request.get('length', 'medium')  # short, medium, long
    style = request.get('style', 'professional')  # professional, casual, academic
    target_audience = request.get('target_audience', 'general')
    
    # Generate a basic outline based on the request
    if length == 'short':
        sections = 3
    elif length == 'long':
        sections = 7
    else:
        sections = 5  # medium
    
    outline = [f"Section {i+1}: Introduction to {topic}" if i == 0 else 
               f"Section {i+1}: Main points about {topic}" if i < sections-1 else
               f"Section {i+1}: Conclusion and next steps"] for i in range(sections)]
    
    return {
        'content_outline': outline,
        'research_notes': [f"Need to research: {topic}", f"Target audience: {target_audience}", f"Style requirements: {style}"]
    }

# Conduct research for content creation
def conduct_research(state: ContentGenerationState):
    request = state['content_request']
    topic = request.get('topic', 'General Topic')
    
    # Simulate research process
    research_data = [
        f"Key finding 1 about {topic}: Importance and relevance",
        f"Key finding 2 about {topic}: Current trends and statistics",
        f"Key finding 3 about {topic}: Future implications and opportunities"
    ]
    
    updated_notes = state.get('research_notes', []) + research_data
    
    return {
        'research_notes': updated_notes
    }

# Generate initial draft content
def generate_draft_content(state: ContentGenerationState):
    request = state['content_request']
    topic = request.get('topic', 'General Topic')
    length = request.get('length', 'medium')
    style = request.get('style', 'professional')
    outline = state['content_outline']
    research_notes = state['research_notes']
    
    # Generate content based on outline and research
    draft = f"# Article on {topic}\n\n"
    
    for i, section in enumerate(outline):
        draft += f"## {section}\n\n"
        draft += f"This section covers {section.lower()}. Based on our research, {research_notes[0].lower()}. "
        
        if i == 0:  # Introduction
            draft += f"{topic} is an important subject that affects many people. "
        elif i == len(outline) - 1:  # Conclusion
            draft += f"In conclusion, {topic} represents significant opportunities for growth and improvement. "
        else:
            draft += f"Further exploration of {topic} reveals interesting patterns and insights. "
        
        draft += "\n\n"
    
    draft += f"Overall, understanding {topic} is crucial for success in today's environment."
    
    return {
        'draft_content': draft
    }

# Evaluate content quality
def evaluate_quality(state: ContentGenerationState):
    draft = state['draft_content']
    request = state['content_request']
    
    # Simple quality assessment
    length_score = min(len(draft) / 1000, 1.0)  # Normalize to 0-1 scale
    keyword_density = draft.lower().count(request.get('topic', '').lower()) / len(draft.split())
    readability_score = 0.8  # Placeholder
    
    # Calculate overall quality score
    quality_score = (length_score + keyword_density * 10 + readability_score) / 3
    quality_score = min(quality_score, 1.0)  # Cap at 1.0
    
    # Generate revision notes based on quality assessment
    revision_notes = []
    if quality_score < 0.5:
        revision_notes.append("Content needs significant improvements in structure and clarity")
    elif quality_score < 0.7:
        revision_notes.append("Content is acceptable but could benefit from additional details")
    else:
        revision_notes.append("Content quality is good, minor refinements may be needed")
    
    return {
        'quality_score': quality_score,
        'revision_notes': revision_notes
    }

# Revise content based on quality feedback
def revise_content(state: ContentGenerationState):
    draft = state['draft_content']
    quality_score = state['quality_score']
    revision_notes = state['revision_notes']
    
    # If quality is good, just make minor revisions
    if quality_score >= 0.7:
        revised_content = draft + "\n\n" + "This content has been reviewed and refined for clarity and accuracy."
    else:
        # Make more substantial revisions
        revised_content = draft.replace(topic := state['content_request'].get('topic', 'General Topic'), 
                                       f"Significant aspects of {topic}")
        revised_content += "\n\n" + "This content has been substantially revised to improve quality and address initial concerns."
    
    return {
        'final_content': revised_content
    }

# Submit content for approval
def submit_for_approval(state: ContentGenerationState):
    final_content = state['final_content']
    quality_score = state['quality_score']
    
    # Determine approval status based on quality score
    if quality_score >= 0.7:
        approval_status = 'approved'
        comments = ['Content meets quality standards', 'Approved for publication']
    elif quality_score >= 0.5:
        approval_status = 'requires_revision'
        comments = ['Content needs minor revisions before approval', 'See revision notes above']
    else:
        approval_status = 'rejected'
        comments = ['Content does not meet quality standards', 'Requires major revisions']
    
    return {
        'approval_status': approval_status,
        'reviewer_comments': comments
    }

# Build the content generation workflow graph
def create_content_generation_graph():
    workflow = StateGraph(ContentGenerationState)
    
    # Add nodes
    workflow.add_node('parse_content_request', parse_content_request)
    workflow.add_node('conduct_research', conduct_research)
    workflow.add_node('generate_draft_content', generate_draft_content)
    workflow.add_node('evaluate_quality', evaluate_quality)
    workflow.add_node('revise_content', revise_content)
    workflow.add_node('submit_for_approval', submit_for_approval)
    
    # Set entry point
    workflow.set_entry_point('parse_content_request')
    
    # Add edges
    workflow.add_edge('parse_content_request', 'conduct_research')
    workflow.add_edge('conduct_research', 'generate_draft_content')
    workflow.add_edge('generate_draft_content', 'evaluate_quality')
    workflow.add_edge('evaluate_quality', 'revise_content')
    workflow.add_edge('revise_content', 'submit_for_approval')
    workflow.add_edge('submit_for_approval', END)
    
    return workflow.compile()

In [12]:
# Test the content generation workflow
content_generation_graph = create_content_generation_graph()

# Example: Generate content about AI in healthcare
initial_state_content = {
    'content_request': {
        'topic': 'artificial intelligence in healthcare',
        'length': 'medium',
        'style': 'professional',
        'target_audience': 'healthcare professionals'
    },
    'generated_content': '',
    'content_outline': [],
    'research_notes': [],
    'draft_content': '',
    'quality_score': 0.0,
    'revision_notes': [],
    'final_content': '',
    'approval_status': '',
    'reviewer_comments': []
}

result_content = content_generation_graph.invoke(initial_state_content)
print("Final content generation state:", result_content)

Final content generation state: {'content_request': {'topic': 'artificial intelligence in healthcare', 'length': 'medium', 'style': 'professional', 'target_audience': 'healthcare professionals'}, 'generated_content': '', 'content_outline': ['Section 1: Introduction to artificial intelligence in healthcare', 'Section 2: Main points about artificial intelligence in healthcare', 'Section 3: Main points about artificial intelligence in healthcare', 'Section 4: Main points about artificial intelligence in healthcare', 'Section 5: Conclusion and next steps'], 'research_notes': ['Need to research: artificial intelligence in healthcare', 'Target audience: healthcare professionals', 'Style requirements: professional', 'Key finding 1 about artificial intelligence in healthcare: Importance and relevance', 'Key finding 2 about artificial intelligence in healthcare: Current trends and statistics', 'Key finding 3 about artificial intelligence in healthcare: Future implications and opportunities'], '

## Summary

In this hands-on project tutorial, we built several practical applications with LangGraph:

1. **Customer Support Chatbot**: A contextual support bot that classifies intents, retrieves user context, and handles various types of customer issues with escalation capabilities.

2. **Research Assistant**: An AI researcher that plans investigations, executes searches, evaluates source credibility, synthesizes information, and generates reports.

3. **Document Analysis Pipeline**: A system that ingests documents, classifies them, extracts entities, validates information, routes appropriately, and generates summaries.

4. **Task Management System**: An intelligent system that parses requests, prioritizes tasks, assigns resources, schedules work, tracks progress, and generates reports.

5. **Content Generation Workflow**: An automated pipeline that takes content requests, conducts research, generates drafts, evaluates quality, revises content, and manages approval workflows.

These projects demonstrate how LangGraph can be used to build sophisticated, real-world applications that handle complex workflows with multiple steps, decision points, and feedback loops.